# Spark 实时 GMV

从 Kafka 读取订单，按门店和 5 分钟窗口计算 GMV。

## 运行条件

运行需要 Spark、Java、Kafka 连接器和可访问的 Kafka，配置见 [[processing/work/spark/README|Spark 运行方式]]。事件包含 `event_time`、`store_id`、`order_id`、`amount`，时间示例为 `2026-07-10 10:00:00`。将 `start_stream` 改为 `True` 后阻塞等待，中断内核会停止查询。首次启动从最新偏移量读取，启动后发送事件并观察控制台窗口汇总。Checkpoint 应为该查询独享的持久目录，`/tmp` 仅用于本地练习。


In [ ]:
from __future__ import annotations
import argparse


## 实现


In [ ]:
def build_query(bootstrap_servers: str, topic: str, checkpoint: str):
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import col, from_json, sum as spark_sum, to_timestamp, window
    from pyspark.sql.types import DoubleType, StringType, StructField, StructType

    spark = SparkSession.builder.appName("store-gmv-streaming").getOrCreate()
    schema = StructType(
        [
            StructField("event_time", StringType(), False),
            StructField("store_id", StringType(), False),
            StructField("order_id", StringType(), False),
            StructField("amount", DoubleType(), False),
        ]
    )

    raw = (
        spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", bootstrap_servers)
        .option("subscribe", topic)
        .option("startingOffsets", "latest")
        .load()
    )

    events = (
        raw.select(from_json(col("value").cast("string"), schema).alias("data"))
        .select("data.*")
        .withColumn("event_ts", to_timestamp("event_time"))
    )

    gmv = (
        events.withWatermark("event_ts", "10 minutes")
        .groupBy(window("event_ts", "5 minutes"), "store_id")
        .agg(spark_sum("amount").alias("gmv"))
    )

    return (
        gmv.writeStream.outputMode("update")
        .format("console")
        .option("truncate", "false")
        .option("checkpointLocation", checkpoint)
        .start()
    )


## 连接服务

先修改地址与参数，再开启执行开关。


In [ ]:
bootstrap_servers = "localhost:9092"
topic = "store_orders"
checkpoint = "/tmp/processing-store-gmv"
start_stream = False

if start_stream:
    query = build_query(bootstrap_servers, topic, checkpoint)
    try:
        query.awaitTermination()
    finally:
        query.stop()


## 命令行入口

导出为脚本后使用；在 Notebook 内运行仅定义入口，不解析内核参数。


In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--bootstrap", required=True)
    parser.add_argument("--topic", default="store_orders")
    parser.add_argument("--checkpoint", required=True)
    args = parser.parse_args()
    query = build_query(args.bootstrap, args.topic, args.checkpoint)
    query.awaitTermination()


if __name__ == "__main__" and "get_ipython" not in globals():
    main()
